# Model 2 Training - YOLOv8s (Small) - Google Colab

Train YOLOv8s model on Google Colab with GPU:
1. **Baseline** - No augmentation (on processed data)
2. **Augmented** - With on-the-fly augmentation (on processed data)

Then evaluate both models on the test set and generate comprehensive results.

**Note: Make sure `yolo_dataset.zip` is in `/content/drive/MyDrive/521_final_plroject/`**


## Step 1: Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted successfully!')


## Step 2: Check GPU


In [ ]:
import torch
print('='*70)
print('GPU DIAGNOSIS')
print('='*70)
print(f'PyTorch CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'PyTorch CUDA version: {torch.version.cuda}')
    print(f'PyTorch device count: {torch.cuda.device_count()}')
    print(f'GPU Device: {torch.cuda.get_device_name(0)}')
    print('SUCCESS: GPU DETECTED!')
    device = 0
else:
    print('ERROR: NO GPU DETECTED!')
    print('Please enable GPU: Runtime -> Change runtime type -> T4 GPU')
    device = 'cpu'
print(f'YOLO Device: {device}')
print('='*70)


## Step 3: Extract Dataset from Zip


In [ ]:
# Extract dataset from Drive to Colab local storage (faster training)
zip_path = '/content/drive/MyDrive/521_final_plroject/yolo_dataset.zip'

print('Extracting dataset to /content/ for faster access...')
print('This may take 5-10 minutes...')

!unzip -q {zip_path} -d /content/

# Verify
!ls /content/data/processed/yolo_format/
print('\nDataset extracted to /content/data/processed/yolo_format/')


## Step 4: Update Dataset YAML Paths


In [ ]:
import yaml
from pathlib import Path

# Update dataset.yaml paths for Colab
colab_data_path = Path('/content/data/processed/yolo_format')
dataset_yaml_path = colab_data_path / 'dataset.yaml'

# Load and update the dataset.yaml file
with open(dataset_yaml_path, 'r') as f:
    dataset_config = yaml.safe_load(f)

# Update paths
dataset_config['path'] = str(colab_data_path)
dataset_config['train'] = 'images/train'
dataset_config['val'] = 'images/val'
if 'test' in dataset_config and dataset_config['test']:
    dataset_config['test'] = 'images/test'

# Save the updated dataset.yaml
with open(dataset_yaml_path, 'w') as f:
    yaml.safe_dump(dataset_config, f, default_flow_style=False)

print(f'Updated dataset.yaml at {dataset_yaml_path} with Colab paths')


## Step 5: Install Packages


In [ ]:
%pip install ultralytics -q
print('Packages installed!')


## Step 6: Configuration


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ultralytics import YOLO
import yaml
import shutil
import warnings
warnings.filterwarnings('ignore')

# Set style for visualizations
plt.style.use('default')
sns.set_palette("husl")

# Model Configuration
MODEL_VARIANT = 'yolov8s.pt'
MODEL_NAME = 'YOLOv8s'

CONFIG = {
    'epochs': 50,
    'batch_size': 16,
    'img_size': 640,
    'patience': 10,
    'device': device,  # Use GPU from Step 2
}

# Augmentation settings (on-the-fly)
AUGMENTATION_CONFIG = {
    'hsv_h': 0.015,
    'hsv_s': 0.7,
    'hsv_v': 0.4,
    'translate': 0.1,
    'scale': 0.5,
    'fliplr': 0.5,
    'mosaic': 1.0,
}

# Paths - Data from /content/, Results to Drive
YOLO_DIR = Path('/content/data/processed/yolo_format')
DATASET_YAML = YOLO_DIR / 'dataset.yaml'
DATASET_YAML_BACKUP = YOLO_DIR / 'dataset.yaml.backup'

# Save results to Drive (will persist after Colab disconnects)
DRIVE_PROJECT = Path('/content/drive/MyDrive/521_final_plroject')
RESULTS_DIR = DRIVE_PROJECT / 'results/models_output'
VIZ_DIR = DRIVE_PROJECT / 'results/visualizations/training'
TEST_RESULTS_DIR = DRIVE_PROJECT / 'results/test_evaluation'
TEST_VIZ_DIR = DRIVE_PROJECT / 'results/visualizations/test_evaluation'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
VIZ_DIR.mkdir(parents=True, exist_ok=True)
TEST_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
TEST_VIZ_DIR.mkdir(parents=True, exist_ok=True)

# Verify dataset.yaml exists
if not DATASET_YAML.exists():
    raise FileNotFoundError(f'Dataset YAML not found at {DATASET_YAML}')

print('Configuration:')
print(f'  Model: {MODEL_NAME} ({MODEL_VARIANT})')
print(f'  Epochs: {CONFIG["epochs"]}')
print(f'  Batch Size: {CONFIG["batch_size"]}')
print(f'  Device: {CONFIG["device"]}')
print(f'  Dataset: {DATASET_YAML}')
print(f'  Results save to Drive: {RESULTS_DIR}')


## Step 7: Verify Dataset


In [ ]:
with open(DATASET_YAML, 'r') as f:
    dataset_config = yaml.safe_load(f)

train_count = len(list((YOLO_DIR / 'images/train').glob('*.jpg')))
val_count = len(list((YOLO_DIR / 'images/val').glob('*.jpg')))
test_count = len(list((YOLO_DIR / 'images/test').glob('*.jpg')))

print('Dataset Information:')
print(f'  Classes: {dataset_config["nc"]}')
print(f'  Train: {train_count:,} images')
print(f'  Val: {val_count:,} images')
print(f'  Test: {test_count:,} images')


## Step 8: Train Baseline Model (No Augmentation)


In [ ]:
print('='*70)
print(f'TRAINING {MODEL_NAME} BASELINE MODEL (NO AUGMENTATION)')
print('='*70)

model_baseline = YOLO(MODEL_VARIANT)

results_baseline = model_baseline.train(
    data=str(DATASET_YAML),
    epochs=CONFIG['epochs'],
    imgsz=CONFIG['img_size'],
    batch=CONFIG['batch_size'],
    device=CONFIG['device'],
    patience=CONFIG['patience'],
    project=str(RESULTS_DIR),
    name=f'{MODEL_NAME.lower()}_baseline',
    # Disable all augmentation
    hsv_h=0.0, hsv_s=0.0, hsv_v=0.0,
    degrees=0.0, translate=0.0, scale=0.0,
    fliplr=0.0, mosaic=0.0,
    save=True,
    plots=True,
    verbose=True
)

# Evaluate on validation set
metrics_baseline = model_baseline.val()

print(f'\n{MODEL_NAME} Baseline Training Complete!')
print(f'  mAP@0.5: {metrics_baseline.box.map50:.4f}')
print(f'  mAP@0.5:0.95: {metrics_baseline.box.map:.4f}')
print(f'  Precision: {metrics_baseline.box.mp:.4f}')
print(f'  Recall: {metrics_baseline.box.mr:.4f}')
print(f'  Best model: {RESULTS_DIR}/{MODEL_NAME.lower()}_baseline/weights/best.pt')


## Step 9: Train Augmented Model (On-the-Fly Augmentation)


In [ ]:
print('='*70)
print(f'TRAINING {MODEL_NAME} WITH AUGMENTATION (ON-THE-FLY)')
print('='*70)

model_augmented = YOLO(MODEL_VARIANT)

results_augmented = model_augmented.train(
    data=str(DATASET_YAML),
    epochs=CONFIG['epochs'],
    imgsz=CONFIG['img_size'],
    batch=CONFIG['batch_size'],
    device=CONFIG['device'],
    patience=CONFIG['patience'],
    project=str(RESULTS_DIR),
    name=f'{MODEL_NAME.lower()}_augmented',
    # Enable augmentation
    hsv_h=AUGMENTATION_CONFIG['hsv_h'],
    hsv_s=AUGMENTATION_CONFIG['hsv_s'],
    hsv_v=AUGMENTATION_CONFIG['hsv_v'],
    translate=AUGMENTATION_CONFIG['translate'],
    scale=AUGMENTATION_CONFIG['scale'],
    fliplr=AUGMENTATION_CONFIG['fliplr'],
    mosaic=AUGMENTATION_CONFIG['mosaic'],
    save=True,
    plots=True,
    verbose=True
)

# Evaluate on validation set
metrics_augmented = model_augmented.val()

print(f'\n{MODEL_NAME} Augmented Training Complete!')
print(f'  mAP@0.5: {metrics_augmented.box.map50:.4f}')
print(f'  mAP@0.5:0.95: {metrics_augmented.box.map:.4f}')
print(f'  Precision: {metrics_augmented.box.mp:.4f}')
print(f'  Recall: {metrics_augmented.box.mr:.4f}')
print(f'  Best model: {RESULTS_DIR}/{MODEL_NAME.lower()}_augmented/weights/best.pt')


## Step 10: Validation Set Performance Comparison


In [ ]:
# Create comparison DataFrame
comparison_val = pd.DataFrame({
    'Model': ['Baseline (Val)', 'Augmented (Val)'],
    'mAP@0.5': [metrics_baseline.box.map50, metrics_augmented.box.map50],
    'mAP@0.5:0.95': [metrics_baseline.box.map, metrics_augmented.box.map],
    'Precision': [metrics_baseline.box.mp, metrics_augmented.box.mp],
    'Recall': [metrics_baseline.box.mr, metrics_augmented.box.mr]
})

print('='*70)
print(f'{MODEL_NAME} VALIDATION SET PERFORMANCE COMPARISON')
print('='*70)
print(comparison_val.to_string(index=False))

improvement_map50 = (metrics_augmented.box.map50 - metrics_baseline.box.map50) / metrics_baseline.box.map50 * 100
improvement_map = (metrics_augmented.box.map - metrics_baseline.box.map) / metrics_baseline.box.map * 100

print(f'\nImprovement (mAP@0.5): {improvement_map50:+.2f}%')
print(f'Improvement (mAP@0.5:0.95): {improvement_map:+.2f}%')

# Save comparison
comparison_val.to_csv(RESULTS_DIR / f'{MODEL_NAME.lower()}_validation_comparison.csv', index=False)
print(f'\nSaved to: {RESULTS_DIR}/{MODEL_NAME.lower()}_validation_comparison.csv')


## Step 11: Plot Training Curves


In [ ]:
# Load training results CSV
baseline_hist = pd.read_csv(RESULTS_DIR / f'{MODEL_NAME.lower()}_baseline/results.csv')
augmented_hist = pd.read_csv(RESULTS_DIR / f'{MODEL_NAME.lower()}_augmented/results.csv')

baseline_hist.columns = baseline_hist.columns.str.strip()
augmented_hist.columns = augmented_hist.columns.str.strip()

# Create comprehensive training curves comparison
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(f'{MODEL_NAME} Training Curves: Baseline vs Augmented', fontsize=16, fontweight='bold')

metrics_to_plot = [
    ('train/box_loss', 'Train Loss', axes[0, 0]),
    ('val/box_loss', 'Val Loss', axes[0, 1]),
    ('metrics/mAP50(B)', 'mAP@0.5', axes[0, 2]),
    ('metrics/mAP50-95(B)', 'mAP@0.5:0.95', axes[1, 0]),
    ('metrics/precision(B)', 'Precision', axes[1, 1]),
    ('metrics/recall(B)', 'Recall', axes[1, 2])
]

for metric_col, title, ax in metrics_to_plot:
    if metric_col in baseline_hist.columns:
        ax.plot(baseline_hist['epoch'], baseline_hist[metric_col], label='Baseline', lw=2, marker='o', markersize=3)
    if metric_col in augmented_hist.columns:
        ax.plot(augmented_hist['epoch'], augmented_hist[metric_col], label='Augmented', lw=2, marker='s', markersize=3)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch', fontsize=10)
    ax.set_ylabel(title, fontsize=10)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(VIZ_DIR / f'{MODEL_NAME.lower()}_training_curves.png', dpi=300, bbox_inches='tight')
plt.savefig(VIZ_DIR / f'{MODEL_NAME.lower()}_training_curves.pdf', bbox_inches='tight')
print(f'Saved to: {VIZ_DIR}/{MODEL_NAME.lower()}_training_curves.png')
plt.show()


## Step 12: Prepare Dataset YAML for Test Evaluation


In [ ]:
# Load original dataset config
with open(DATASET_YAML, 'r') as f:
    dataset_config = yaml.safe_load(f)

# Backup original dataset.yaml
if DATASET_YAML_BACKUP.exists():
    shutil.copy(str(DATASET_YAML_BACKUP), str(DATASET_YAML))
else:
    shutil.copy(str(DATASET_YAML), str(DATASET_YAML_BACKUP))

# Modify dataset.yaml to use test split for validation
original_val = dataset_config.get('val', 'images/val')
original_test = dataset_config.get('test', 'images/test')

test_dataset_config = dataset_config.copy()
test_dataset_config['val'] = original_test  # Use test as val for evaluation

# Save modified config
with open(DATASET_YAML, 'w') as f:
    yaml.safe_dump(test_dataset_config, f, default_flow_style=False)

print('Dataset YAML temporarily modified to use test split for evaluation')
print(f'  Original val: {original_val}')
print(f'  Now using as val (test set): {original_test}')


## Step 13: Evaluate Baseline Model on Test Set


In [ ]:
print('='*70)
print(f'EVALUATING {MODEL_NAME} BASELINE MODEL ON TEST SET')
print('='*70)

# Load baseline model
baseline_model_path = RESULTS_DIR / f'{MODEL_NAME.lower()}_baseline' / 'weights' / 'best.pt'
model_baseline_test = YOLO(str(baseline_model_path))

# Evaluate on test set
test_metrics_baseline = model_baseline_test.val(
    data=str(DATASET_YAML),
    imgsz=640,
    conf=0.25,
    iou=0.7,
    save_json=True,
    save_hybrid=False,
    plots=True,
    project=str(TEST_RESULTS_DIR),
    name=f'{MODEL_NAME.lower()}_baseline_test'
)

print(f'\n{MODEL_NAME} Baseline Test Set Performance:')
print(f'  mAP@0.5: {test_metrics_baseline.box.map50:.4f}')
print(f'  mAP@0.5:0.95: {test_metrics_baseline.box.map:.4f}')
print(f'  Precision: {test_metrics_baseline.box.mp:.4f}')
print(f'  Recall: {test_metrics_baseline.box.mr:.4f}')
print(f'\nResults saved to: {TEST_RESULTS_DIR}/{MODEL_NAME.lower()}_baseline_test/')


## Step 14: Evaluate Augmented Model on Test Set


In [ ]:
print('='*70)
print(f'EVALUATING {MODEL_NAME} AUGMENTED MODEL ON TEST SET')
print('='*70)

# Load augmented model
augmented_model_path = RESULTS_DIR / f'{MODEL_NAME.lower()}_augmented' / 'weights' / 'best.pt'
model_augmented_test = YOLO(str(augmented_model_path))

# Evaluate on test set
test_metrics_augmented = model_augmented_test.val(
    data=str(DATASET_YAML),
    imgsz=640,
    conf=0.25,
    iou=0.7,
    save_json=True,
    save_hybrid=False,
    plots=True,
    project=str(TEST_RESULTS_DIR),
    name=f'{MODEL_NAME.lower()}_augmented_test'
)

print(f'\n{MODEL_NAME} Augmented Test Set Performance:')
print(f'  mAP@0.5: {test_metrics_augmented.box.map50:.4f}')
print(f'  mAP@0.5:0.95: {test_metrics_augmented.box.map:.4f}')
print(f'  Precision: {test_metrics_augmented.box.mp:.4f}')
print(f'  Recall: {test_metrics_augmented.box.mr:.4f}')
print(f'\nResults saved to: {TEST_RESULTS_DIR}/{MODEL_NAME.lower()}_augmented_test/')


## Step 15: Restore Original Dataset YAML


In [ ]:
# Restore original dataset.yaml
shutil.copy(str(DATASET_YAML_BACKUP), str(DATASET_YAML))
print('Original dataset.yaml restored')


## Step 16: Test Set Performance Comparison


In [ ]:
# Create comparison DataFrame
comparison_test = pd.DataFrame({
    'Model': ['Baseline (Test)', 'Augmented (Test)'],
    'mAP@0.5': [test_metrics_baseline.box.map50, test_metrics_augmented.box.map50],
    'mAP@0.5:0.95': [test_metrics_baseline.box.map, test_metrics_augmented.box.map],
    'Precision': [test_metrics_baseline.box.mp, test_metrics_augmented.box.mp],
    'Recall': [test_metrics_baseline.box.mr, test_metrics_augmented.box.mr]
})

print('='*70)
print(f'{MODEL_NAME} TEST SET PERFORMANCE COMPARISON')
print('='*70)
print(comparison_test.to_string(index=False))

# Calculate improvement
test_improvement_map50 = (test_metrics_augmented.box.map50 - test_metrics_baseline.box.map50) / test_metrics_baseline.box.map50 * 100
test_improvement_map = (test_metrics_augmented.box.map - test_metrics_baseline.box.map) / test_metrics_baseline.box.map * 100

print(f'\nTest Set Improvement (mAP@0.5): {test_improvement_map50:+.2f}%')
print(f'Test Set Improvement (mAP@0.5:0.95): {test_improvement_map:+.2f}%')

# Save comparison
comparison_test.to_csv(TEST_RESULTS_DIR / f'{MODEL_NAME.lower()}_test_comparison.csv', index=False)
print(f'\nSaved to: {TEST_RESULTS_DIR}/{MODEL_NAME.lower()}_test_comparison.csv')


## Step 17: Visualize Test Set Performance


In [ ]:
# Create visualization comparing test set performance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot for mAP metrics
metrics_to_plot = ['mAP@0.5', 'mAP@0.5:0.95']
x = np.arange(len(metrics_to_plot))
width = 0.35

baseline_values = [test_metrics_baseline.box.map50, test_metrics_baseline.box.map]
augmented_values = [test_metrics_augmented.box.map50, test_metrics_augmented.box.map]

axes[0].bar(x - width/2, baseline_values, width, label='Baseline', alpha=0.8)
axes[0].bar(x + width/2, augmented_values, width, label='Augmented', alpha=0.8)
axes[0].set_ylabel('mAP Score', fontsize=12)
axes[0].set_title(f'{MODEL_NAME} Test Set: mAP Metrics Comparison', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics_to_plot)
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].set_ylim([0, max(max(baseline_values), max(augmented_values)) * 1.2])

# Bar plot for Precision and Recall
pr_metrics = ['Precision', 'Recall']
x2 = np.arange(len(pr_metrics))

baseline_pr = [test_metrics_baseline.box.mp, test_metrics_baseline.box.mr]
augmented_pr = [test_metrics_augmented.box.mp, test_metrics_augmented.box.mr]

axes[1].bar(x2 - width/2, baseline_pr, width, label='Baseline', alpha=0.8)
axes[1].bar(x2 + width/2, augmented_pr, width, label='Augmented', alpha=0.8)
axes[1].set_ylabel('Score', fontsize=12)
axes[1].set_title(f'{MODEL_NAME} Test Set: Precision & Recall Comparison', fontsize=14, fontweight='bold')
axes[1].set_xticks(x2)
axes[1].set_xticklabels(pr_metrics)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].set_ylim([0, max(max(baseline_pr), max(augmented_pr)) * 1.2])

plt.tight_layout()
plt.savefig(TEST_VIZ_DIR / f'{MODEL_NAME.lower()}_test_performance_comparison.png', dpi=300, bbox_inches='tight')
plt.savefig(TEST_VIZ_DIR / f'{MODEL_NAME.lower()}_test_performance_comparison.pdf', bbox_inches='tight')
print(f'Saved to: {TEST_VIZ_DIR}/{MODEL_NAME.lower()}_test_performance_comparison.png')
plt.show()


## Step 18: Summary and Model Locations


In [ ]:
print('='*70)
print(f'{MODEL_NAME} TRAINING AND EVALUATION COMPLETE')
print('='*70)

print('\nVALIDATION SET PERFORMANCE:')
print(f'  Baseline - mAP@0.5: {metrics_baseline.box.map50:.4f}')
print(f'  Augmented - mAP@0.5: {metrics_augmented.box.map50:.4f}')

print('\nTEST SET PERFORMANCE:')
print(f'  Baseline - mAP@0.5: {test_metrics_baseline.box.map50:.4f}')
print(f'  Augmented - mAP@0.5: {test_metrics_augmented.box.map50:.4f}')

print('\nBEST MODEL (Test Set):')
if test_metrics_augmented.box.map50 > test_metrics_baseline.box.map50:
    print(f'  Augmented Model performs better!')
    print(f'  mAP@0.5: {test_metrics_augmented.box.map50:.4f}')
    best_model_path = RESULTS_DIR / f'{MODEL_NAME.lower()}_augmented' / 'weights' / 'best.pt'
else:
    print(f'  Baseline Model performs better!')
    print(f'  mAP@0.5: {test_metrics_baseline.box.map50:.4f}')
    best_model_path = RESULTS_DIR / f'{MODEL_NAME.lower()}_baseline' / 'weights' / 'best.pt'

print('\nMODEL LOCATIONS (for deployment):')
print(f'  Baseline: {RESULTS_DIR}/{MODEL_NAME.lower()}_baseline/weights/best.pt')
print(f'  Augmented: {RESULTS_DIR}/{MODEL_NAME.lower()}_augmented/weights/best.pt')
print(f'  Best Model: {best_model_path}')

print('\nOUTPUTS SAVED:')
print(f'  Validation Comparison: {RESULTS_DIR}/{MODEL_NAME.lower()}_validation_comparison.csv')
print(f'  Test Comparison: {TEST_RESULTS_DIR}/{MODEL_NAME.lower()}_test_comparison.csv')
print(f'  Training Curves: {VIZ_DIR}/{MODEL_NAME.lower()}_training_curves.png')
print(f'  Test Performance: {TEST_VIZ_DIR}/{MODEL_NAME.lower()}_test_performance_comparison.png')
print('='*70)


## Step 19: Display Saved Visualizations

The training curves and test performance visualizations are saved in the results folder. Here we display them for reference.

In [ ]:
# Display the saved training curves visualization
from IPython.display import Image, display
from pathlib import Path

# Path to the saved training curves (local repository)
training_curves_path = Path('../results/model_2_visualizations/training/yolov8s_training_curves.png')

if training_curves_path.exists():
    print('YOLOv8s Training Curves: Baseline vs Augmented')
    print('='*60)
    display(Image(filename=str(training_curves_path), width=900))
else:
    print(f'Training curves not found at: {training_curves_path}')
    print('The visualization was generated during Colab training and saved to:')
    print('  results/model_2_visualizations/training/yolov8s_training_curves.png')

In [ ]:
# Display the saved test set performance comparison
test_perf_path = Path('../results/model_2_visualizations/test_evaluation/yolov8s_test_performance_comparison.png')

if test_perf_path.exists():
    print('YOLOv8s Test Set Performance: Baseline vs Augmented')
    print('='*60)
    display(Image(filename=str(test_perf_path), width=900))
else:
    print(f'Test performance comparison not found at: {test_perf_path}')
    print('The visualization was generated during Colab training and saved to:')
    print('  results/model_2_visualizations/test_evaluation/yolov8s_test_performance_comparison.png')